In [ ]:
#1. Parsing. Erfassung von Stellenangeboten im Logistikbereich von Arbeitsagentur.de
import requests
import pandas as pd
import time
import random
import re
from collections import Counter
from bs4 import BeautifulSoup

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/121.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36",
]

def clean_text(text):
    """Entfernt illegale Zeichen die CSV/Excel nicht unterstützt"""
    if not text:
        return ""
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', str(text))

def get_vacancies(keyword, location="Deutschland", pages=100):
    all_jobs = []
    headers = {
        "User-Agent": "Mozilla/5.0",
        "X-API-Key": "jobboerse-jobsuche",
    }
    for page in range(1, pages + 1):
        url = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/jobs"
        params = {
            "was": keyword,
            "wo": location,
            "page": page,
            "size": 25,
            "angebotsart": 1,
        }
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            print(f"  Fehler auf Seite {page}: {response.status_code}")
            break
        data = response.json()
        jobs = data.get("stellenangebote", [])
        if not jobs:
            print(f"  Seite {page}: keine weiteren Stellen — Suche beendet")
            break
        for job in jobs:
            all_jobs.append({
                "Keyword":         keyword,
                "Titel":           job.get("titel", ""),
                "Arbeitgeber":     job.get("arbeitgeber", ""),
                "Ort":             job.get("arbeitsort", {}).get("ort", ""),
                "Bundesland":      job.get("arbeitsort", {}).get("region", ""),
                "Veroeffentlicht": job.get("aktuelleVeroeffentlichungsdatum", ""),
                "Eintrittsdatum":  job.get("eintrittsdatum", ""),
                "Befristung":      job.get("befristung", ""),
                "ExterneUrl":      job.get("externeUrl", ""),
            })
        print(f"  Seite {page}: {len(jobs)} Stellen geladen")
        time.sleep(1)
    return all_jobs


def get_job_description(url, retries=3):
    if not url:
        return "", "", ""
    for attempt in range(retries):
        try:
            headers = {
                "User-Agent": random.choice(USER_AGENTS),
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
                "Accept-Language": "de-DE,de;q=0.9,en;q=0.8",
                "Accept-Encoding": "gzip, deflate, br",
                "Connection": "keep-alive",
            }
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code == 403:
                print(f"  Blockiert (403) — warte und versuche erneut...")
                time.sleep(5 + random.uniform(1, 3))
                continue
            if response.status_code != 200:
                print(f"  Fehler {response.status_code} — übersprungen")
                return "", "", ""
            soup = BeautifulSoup(response.text, "html.parser")
            for tag in soup(["script", "style", "nav", "footer", "header"]):
                tag.decompose()
            full_text = soup.get_text(separator="\n", strip=True)
            if len(full_text) < 200:
                print(f"  Zu wenig Text — möglicherweise blockiert")
                return "", "", ""
            aufgaben = ""
            for marker in ["Aufgaben", "Tätigkeiten", "Ihre Aufgaben",
                           "Deine Aufgaben", "Was Sie erwartet"]:
                if marker in full_text:
                    start = full_text.find(marker)
                    aufgaben = clean_text(full_text[start:start+1500].strip())
                    break
            profil = ""
            for marker in ["Profil", "Anforderungen", "Qualifikation",
                           "Voraussetzungen", "Ihr Profil", "Dein Profil",
                           "Was Sie mitbringen", "Was wir erwarten"]:
                if marker in full_text:
                    start = full_text.find(marker)
                    profil = clean_text(full_text[start:start+1500].strip())
                    break
            beschaeftigung = ""
            for marker in ["Vollzeit", "Teilzeit", "Remote", "Homeoffice", "Hybrid"]:
                if marker in full_text:
                    beschaeftigung += marker + " "
            return aufgaben, profil, beschaeftigung.strip()
        except Exception as e:
            print(f"  Fehler (Versuch {attempt+1}/{retries}): {e}")
            time.sleep(3)
    return "", "", ""


keywords = [
    "Logistiker",
    "Logistikmanager",
    "Logistics Manager",
    "Logistikleiter",
    "Leiter Logistik",
    "Logistikkoordinator",
    "Logistikfachkraft",
]

# ============================================================
# SCHRITT 1: STELLENANGEBOTE SAMMELN
# ============================================================
print("=== Logistiker: Stellenangebote werden gesammelt ===\n")
all_vacancies = []

for kw in keywords:
    print(f"Suche nach: {kw}")
    results = get_vacancies(keyword=kw, pages=100)
    all_vacancies.extend(results)
    print(f"  Gesamt für '{kw}': {len(results)} Stellen\n")
    time.sleep(2)

# ============================================================
# SCHRITT 2: DEDUPLIZIERUNG
# ============================================================
df = pd.DataFrame(all_vacancies)
before = len(df)
df = df.drop_duplicates(subset=["Titel", "Arbeitgeber", "Ort"])
after = len(df)
print(f"Deduplizierung: {before} → {after} eindeutige Stellen")

# ============================================================
# SCHRITT 3: NORMALISIERUNG
# ============================================================
df["Titel"]       = df["Titel"].astype(str).str.strip()
df["Arbeitgeber"] = df["Arbeitgeber"].astype(str).str.strip()
df["Ort"]         = df["Ort"].astype(str).str.strip()
df["Bundesland"]  = df["Bundesland"].astype(str).str.strip()
print("Normalisierung: fertig")

# ============================================================
# SCHRITT 4: DATUMSFORMAT
# ============================================================
df["Veroeffentlicht"] = pd.to_datetime(
    df["Veroeffentlicht"], errors="coerce"
).dt.strftime("%Y-%m-%d")
df["Eintrittsdatum"] = pd.to_datetime(
    df["Eintrittsdatum"], errors="coerce"
).dt.strftime("%Y-%m-%d")
print("Datumsformat: fertig")

# ============================================================
# SCHRITT 5: LEERE WERTE
# ============================================================
df = df.replace(["", "None", "nan", "null", "NaN"], pd.NA)
print("Leere Werte: bereinigt")

# ============================================================
# SCHRITT 6: ERFAHRUNGSNIVEAU
# ============================================================
def classify_level(titel):
    titel = str(titel).lower()
    if any(w in titel for w in ["junior", "einsteiger", "berufseinsteiger"]):
        return "Junior"
    elif any(w in titel for w in ["senior", "lead", "head", "leiter", "director"]):
        return "Senior"
    elif any(w in titel for w in ["manager", "koordinator", "spezialist"]):
        return "Mid"
    else:
        return "Nicht angegeben"

df["Level"] = df["Titel"].apply(classify_level)
print("Level-Klassifizierung: fertig")

# ============================================================
# SCHRITT 7: STELLENBESCHREIBUNGEN PARSEN
# ============================================================
df_with_url    = df[df["ExterneUrl"].astype(str).str.startswith("http")].copy()
df_without_url = df[~df["ExterneUrl"].astype(str).str.startswith("http")].copy()

print(f"\nMit externer URL: {len(df_with_url)} → werden geparst")
print(f"Ohne URL: {len(df_without_url)} → werden übersprungen")
print("\n=== Stellenbeschreibungen werden geladen ===\n")

aufgaben_list, profil_list, beschaeftigung_list = [], [], []

for i, (idx, row) in enumerate(df_with_url.iterrows()):
    print(f"  [{i+1}/{len(df_with_url)}] {row['Titel']} — {row['Arbeitgeber']}")
    a, p, b = get_job_description(row["ExterneUrl"])
    aufgaben_list.append(a)
    profil_list.append(p)
    beschaeftigung_list.append(b)
    time.sleep(random.uniform(1, 3))

df_with_url["Aufgaben"]       = aufgaben_list
df_with_url["Profil"]         = profil_list
df_with_url["Beschaeftigung"] = beschaeftigung_list

df_without_url["Aufgaben"]       = ""
df_without_url["Profil"]         = ""
df_without_url["Beschaeftigung"] = ""

# ============================================================
# SCHRITT 8: ZUSAMMENFÜHREN UND SPEICHERN
# ============================================================
df_final = pd.concat([df_with_url, df_without_url]).sort_index()
df_final.to_csv("logistiker_jobs_full.csv", index=False, encoding="utf-8-sig")

print(f"\n✓ Fertig! Datei gespeichert: logistiker_jobs_full.csv")
print(f"✓ Stellen gesamt:    {len(df_final)}")
print(f"✓ Mit Beschreibung:  {len(df_with_url)}")
print(f"✓ Ohne Beschreibung: {len(df_without_url)}")
print(f"✓ Level-Verteilung:\n{df_final['Level'].value_counts()}")

=== Logistiker: Stellenangebote werden gesammelt ===

Suche nach: Logistiker
  Seite 1: 25 Stellen geladen
  Seite 2: 25 Stellen geladen
  Seite 3: 25 Stellen geladen
  Seite 4: 25 Stellen geladen
  Seite 5: 25 Stellen geladen
  Seite 6: 25 Stellen geladen
  Seite 7: 25 Stellen geladen
  Seite 8: 25 Stellen geladen
  Seite 9: 25 Stellen geladen
  Seite 10: 25 Stellen geladen
  Seite 11: 25 Stellen geladen
  Seite 12: 25 Stellen geladen
  Seite 13: 25 Stellen geladen
  Seite 14: 25 Stellen geladen
  Seite 15: 25 Stellen geladen
  Seite 16: 25 Stellen geladen
  Seite 17: 25 Stellen geladen
  Seite 18: 25 Stellen geladen
  Seite 19: 25 Stellen geladen
  Seite 20: 25 Stellen geladen
  Seite 21: 25 Stellen geladen
  Seite 22: 25 Stellen geladen
  Seite 23: 25 Stellen geladen
  Seite 24: 25 Stellen geladen
  Seite 25: 25 Stellen geladen
  Seite 26: 25 Stellen geladen
  Seite 27: 25 Stellen geladen
  Seite 28: 25 Stellen geladen
  Seite 29: 25 Stellen geladen
  Seite 30: 25 Stellen geladen
  

In [ ]:
df_final.to_csv("logistiker_jobs_full.csv", index=False, encoding="utf-8-sig")
print("Gespeichert!")

Gespeichert!


In [ ]:
#Extrahieren von Fähigkeiten und Anforderungen aus Stellenanzeigen

TOOLS_KEYWORDS = [
    "SAP", "SAP MM", "SAP WM", "SAP EWM", "SAP TM", "SAP S/4HANA",
    "Oracle", "Microsoft Dynamics", "Navision", "Dynamics 365",
    "TMS", "WMS", "ERP", "LVS", "Warehouse Management",
    "Excel", "MS Office", "Google Sheets", "Word", "PowerPoint",
    "Power BI", "Tableau", "Google Analytics",
    "Cargowise", "Transporeon", "Infor", "Descartes",
    "Jira", "Confluence", "SAP Ariba",
]

HARD_SKILLS_KEYWORDS = [
    "Kommissionierung", "Lagerverwaltung", "Lagerlogistik",
    "Warenwirtschaft", "Inventur", "Bestandsmanagement",
    "Warehousing", "Einlagerung", "Auslagerung", "Wareneingang",
    "Warenausgang", "Verpackung", "Etikettierung",
    "Transportlogistik", "Tourenplanung", "Routenplanung",
    "Auslieferung", "Disposition", "Fuhrparkmanagement",
    "Sendungsverfolgung", "Tracking",
    "Zollabwicklung", "Zoll", "Import", "Export",
    "Frachtbriefe", "Lieferscheine", "Zolldokumente",
    "Lean", "Kaizen", "Prozessoptimierung", "Six Sigma",
    "Qualitätsmanagement", "ISO",
    "Produktionsplanung", "Kapazitätsplanung", "Bedarfsplanung",
    "Materialplanung", "MRP",
    "Reporting", "Analyse", "KPI", "Kennzahlen", "Controlling",
]

BEREICH_KEYWORDS = [
    "Supply Chain", "Spedition", "Frachtmanagement",
    "Einkauf", "Beschaffung", "Lieferantenmanagement",
    "Projektmanagement", "Logistikplanung",
    "Kontraktlogistik", "Fulfillment", "Last Mile",
    "Intralogistik", "Außenhandel", "Seefracht",
    "Luftfracht", "Landtransport", "Kurierdienst",
]

SOFTSKILLS_KEYWORDS = [
    "Teamfähigkeit", "Zuverlässigkeit", "Flexibilität",
    "Belastbarkeit", "körperliche Belastbarkeit",
    "Selbstständigkeit", "Kommunikationsfähigkeit",
    "Dienstleistungsbereitschaft", "Organisationstalent",
    "Eigeninitiative", "Stressresistenz",
    "Durchsetzungsvermögen", "Analytisches Denken",
    "Problemlösungsfähigkeit", "Verhandlungsgeschick",
]

# --- Führerschein — Duplikate zusammengeführt ---
FUEHRERSCHEIN_KEYWORDS = [
    "Führerschein Klasse B",
    "Führerschein Klasse C",
    "Führerschein Klasse C1",
    "Führerschein Klasse CE",
    "Führerschein Klasse BE",
    "Gabelstapler",
]

# --- Sprachen — nur Sprachnamen, ohne beschreibende Formulierungen ---
SPRACHEN_KEYWORDS = [
    "Deutsch", "Englisch", "Französisch", "Spanisch",
    "Polnisch", "Russisch", "Chinesisch", "Italienisch",
    "Niederländisch", "Türkisch",
]

# --- Erfahrung —  Kategorien ohne Duplikate ---
ERFAHRUNG_KEYWORDS = [
    "Berufseinsteiger",
    "Quereinsteiger",
    "Junior",
    "Senior",
    "1 Jahr",
    "2 Jahre",
    "3 Jahre",
    "5 Jahre",
    "mehrjährig",
]

# --- Abschluss — nur die grundlegenden, ohne Kombinationen ---
ABSCHLUSS_KEYWORDS = [
    "Bachelor",
    "Master",
    "MBA",
    "Studium",
    "Ausbildung",
    "Berufsausbildung",
    "Speditionskaufmann",
    "Speditionskauffrau",
    "Fachlagerist",
    "Fachkraft für Lagerlogistik",
    "Industriekaufmann",
    "Industriekauffrau",
    "Kaufmann",
    "Kauffrau",
]

BESCHAEFTIGUNG_KEYWORDS = [
    "Vollzeit", "Teilzeit", "Remote", "Homeoffice",
    "Hybrid", "Minijob", "Werkstudent",
]


# --- Funktionen ---
def find_keywords(text, keywords):
    """Sucht nach Schlüsselwörtern im Text und gibt die gefundenen zurück"""
    if not text or str(text) == "nan":
        return ""
    found = [kw for kw in keywords if kw.lower() in str(text).lower()]
    return ", ".join(found)

def count_keywords(series, top_n=15):
    """Zählt die Häufigkeit jedes einzelnen Keywords"""
    all_keywords = []
    for cell in series.dropna():
        cell_str = str(cell).strip()
        if cell_str and cell_str != "nan":
            keywords = [kw.strip() for kw in cell_str.split(",")]
            keywords = [kw for kw in keywords if kw]
            all_keywords.extend(keywords)
    counter = Counter(all_keywords)
    counter.pop("", None)
    return pd.DataFrame(counter.most_common(top_n), columns=["Keyword", "Count"])

def extract_salary(text):
    """Sucht nach Gehaltsangaben im Stellentitel"""
    if not text or str(text) == "nan":
        return ""
    patterns = [
        r'(\d+[,.]?\d*)\s*€',
        r'(\d+[,.]?\d*)\s*EUR',
        r'(\d+[,.]?\d*)\s*Euro',
    ]
    for pattern in patterns:
        match = re.search(pattern, str(text))
        if match:
            betrag = match.group(1)
            if any(w in str(text) for w in ["Stunde", "Std.", "pro h", "/h", "/ Std"]):
                return f"{betrag} €/Stunde"
            elif any(w in str(text) for w in ["Monat", "mtl.", "monatlich"]):
                return f"{betrag} €/Monat"
            elif any(w in str(text) for w in ["Jahr", "jährlich", "p.a."]):
                return f"{betrag} €/Jahr"
            else:
                return f"{betrag} €"
    return ""

def extract_hours(text):
    """Sucht nach Arbeitsstunden im Stellentitel"""
    if not text or str(text) == "nan":
        return ""
    patterns = [
        r'(\d+)\s*Std\.',
        r'(\d+)\s*h\b',
        r'(\d+)\s*Stunden',
        r'VOLLZEIT\s*(\d+)',
        r'(\d+)\s*Stunden/Woche',
        r'(\d+)\s*h/Woche',
    ]
    for pattern in patterns:
        match = re.search(pattern, str(text))
        if match:
            stunden = match.group(1)
            if any(w in str(text) for w in ["Woche", "wöchentlich", "/Woche"]):
                return f"{stunden} Std/Woche"
            elif any(w in str(text) for w in ["Tag", "täglich", "/Tag"]):
                return f"{stunden} Std/Tag"
            else:
                return f"{stunden} Std/Woche"
    return ""


# --- Datei laden ---
df = pd.read_csv("logistiker_jobs_full.csv", encoding="utf-8-sig")
print(f"Stellen geladen: {len(df)}")

# Aufgaben, Profil und Beschäftigung zu einem Text zusammenführen
df["FullText"] = (
    df["Aufgaben"].astype(str) + " " +
    df["Profil"].astype(str) + " " +
    df["Beschaeftigung"].astype(str)
)

# --- Keywords aus allen Kategorien extrahieren ---
df["Tools"]         = df["FullText"].apply(lambda x: find_keywords(x, TOOLS_KEYWORDS))
df["Hard_Skills"]   = df["FullText"].apply(lambda x: find_keywords(x, HARD_SKILLS_KEYWORDS))
df["Bereich"]       = df["FullText"].apply(lambda x: find_keywords(x, BEREICH_KEYWORDS))
df["Soft_Skills"]   = df["FullText"].apply(lambda x: find_keywords(x, SOFTSKILLS_KEYWORDS))
df["Fuehrerschein"] = df["FullText"].apply(lambda x: find_keywords(x, FUEHRERSCHEIN_KEYWORDS))
df["Sprachen"]      = df["FullText"].apply(lambda x: find_keywords(x, SPRACHEN_KEYWORDS))
df["Erfahrung"]     = df["FullText"].apply(lambda x: find_keywords(x, ERFAHRUNG_KEYWORDS))
df["Abschluss"]     = df["FullText"].apply(lambda x: find_keywords(x, ABSCHLUSS_KEYWORDS))
df["Arbeitszeit"]   = df["FullText"].apply(lambda x: find_keywords(x, BESCHAEFTIGUNG_KEYWORDS))
df["Gehalt"]        = df["Titel"].apply(extract_salary)
df["Arbeitsstunden"]= df["Titel"].apply(extract_hours)

# Hilfsspalte entfernen
df = df.drop(columns=["FullText"])

# Datei speichern
df.to_csv("logistiker_jobs_full_skills.csv", index=False, encoding="utf-8-sig")

# --- Abschlussstatistik ---
print(f"\nFertig! Datei gespeichert: logistiker_jobs_full_skills.csv")
print(f"Mit Tools:          {df['Tools'].astype(str).str.strip().ne('').sum()}")
print(f"Mit Hard Skills:    {df['Hard_Skills'].astype(str).str.strip().ne('').sum()}")
print(f"Mit Soft Skills:    {df['Soft_Skills'].astype(str).str.strip().ne('').sum()}")
print(f"Mit Führerschein:   {df['Fuehrerschein'].astype(str).str.strip().ne('').sum()}")
print(f"Mit Sprachen:       {df['Sprachen'].astype(str).str.strip().ne('').sum()}")
print(f"Mit Arbeitszeit:    {df['Arbeitszeit'].astype(str).str.strip().ne('').sum()}")
print(f"Mit Gehalt:         {df['Gehalt'].astype(str).str.strip().ne('').sum()}")
print(f"Mit Arbeitsstunden: {df['Arbeitsstunden'].astype(str).str.strip().ne('').sum()}")

Stellen geladen: 2867

Fertig! Datei gespeichert: logistiker_jobs_full_skills.csv
Mit Tools:          176
Mit Hard Skills:    186
Mit Soft Skills:    131
Mit Führerschein:   34
Mit Sprachen:       174
Mit Arbeitszeit:    151
Mit Gehalt:         93
Mit Arbeitsstunden: 4


In [ ]:
#Bereinigung und Vorbereitung der endgültigen Datei für die Analyse
# --- Datei laden ---
df = pd.read_csv("logistiker_jobs_full_skills.csv", encoding="utf-8-sig")
print(f"Stellen geladen: {len(df)}")

# --- Hinzufügen ID ---
df.insert(0, "ID", range(1, len(df) + 1))

# --- Nur die relevanten Spalten behalten ---
df_clean = df[[
    "ID",
    "Titel",
    "Arbeitgeber",
    "Ort",
    "Bundesland",
    "Veroeffentlicht",
    "Eintrittsdatum",
    "Befristung",
    "Level",
    "Tools",
    "Hard_Skills",
    "Bereich",
    "Soft_Skills",
    "Fuehrerschein",
    "Sprachen",
    "Erfahrung",
    "Abschluss",
    "Arbeitszeit",
    "Gehalt",
    "Arbeitsstunden",
]].copy()

# --- Gruppierung der Gehälter ---
# Runden Sie den Stundenlohn auf die nächste ganze Zahl.
def normalize_gehalt(text):
    if not text or str(text) == "nan":
        return ""
    text = str(text)
    # die Zahl extrahieren
    import re
    match = re.search(r'(\d+)[,.]?\d*\s*€/(Stunde|Monat|Jahr)', text)
    if match:
        betrag = int(float(match.group(1).replace(",", ".")))
        typ = match.group(2)
        return f"{betrag} €/{typ}"
    return text

df_clean["Gehalt"] = df_clean["Gehalt"].apply(normalize_gehalt)

# --- Datumsfilter – erst ab Juni 2025 ---
df_clean["Veroeffentlicht"] = pd.to_datetime(df_clean["Veroeffentlicht"], errors="coerce")
df_clean = df_clean[df_clean["Veroeffentlicht"] >= "2025-06-01"]
df_clean["Veroeffentlicht"] = df_clean["Veroeffentlicht"].dt.strftime("%Y-%m-%d")

print(f"Nach Datumsfilter: {len(df_clean)} Stellen")

df_clean.to_csv("logistiker_jobs_clean.csv", index=False, encoding="utf-8-sig")
print(f"Gespeichert: logistiker_jobs_clean.csv")

Stellen geladen: 2867
Nach Datumsfilter: 2733 Stellen
Gespeichert: logistiker_jobs_clean.csv


In [ ]:
#Vorbereitung von Daten für die Analyse in Power BI

import pandas as pd
from collections import Counter

# ============================================================
# SCHRITT 1: ALTE STELLEN ENTFERNEN (2021-2024)
# ============================================================
df = pd.read_csv("logistiker_jobs_clean.csv", encoding="utf-8-sig")
print(f"Stellen vorher: {len(df)}")

df["Veroeffentlicht"] = pd.to_datetime(df["Veroeffentlicht"], errors="coerce")

print("\nVerteilung nach Jahren (vorher):")
print(df["Veroeffentlicht"].dt.year.value_counts().sort_index())

# nur ab Juni 2025 behalten
df = df[df["Veroeffentlicht"] >= "2025-06-01"]
df["Veroeffentlicht"] = df["Veroeffentlicht"].dt.strftime("%Y-%m-%d")

print(f"\nStellen nachher: {len(df)}")
print(f"Zeitraum: {df['Veroeffentlicht'].min()} — {df['Veroeffentlicht'].max()}")

df.to_csv("logistiker_jobs_clean.csv", index=False, encoding="utf-8-sig")
print("logistiker_jobs_clean.csv gespeichert!")

# ============================================================
# SCHRITT 2: KEYWORDS TABELLE ERSTELLEN
# ============================================================
def expand_column(df, col, kategorie):
    rows = []
    for i, row in df.iterrows():
        cell = str(row[col]).strip()
        if cell and cell != "nan":
            keywords = [kw.strip() for kw in cell.split(",") if kw.strip()]
            for kw in keywords:
                rows.append({
                    "ID":          row["ID"],
                    "Titel":       row["Titel"],
                    "Arbeitgeber": row["Arbeitgeber"],
                    "Ort":         row["Ort"],
                    "Bundesland":  row["Bundesland"],
                    "Kategorie":   kategorie,
                    "Keyword":     kw,
                })
    return pd.DataFrame(rows)

# Aufschlüsselung aller Kategorien
tools_df       = expand_column(df, "Tools",         "Tools")
skills_df      = expand_column(df, "Hard_Skills",   "Hard_Skills")
bereich_df     = expand_column(df, "Bereich",       "Bereich")
soft_df        = expand_column(df, "Soft_Skills",   "Soft_Skills")
sprachen_df    = expand_column(df, "Sprachen",      "Sprachen")
fuehr_df       = expand_column(df, "Fuehrerschein", "Fuehrerschein")
arbeitszeit_df = expand_column(df, "Arbeitszeit",   "Arbeitszeit")

# Alles zusammenführen
all_keywords = pd.concat([
    tools_df,
    skills_df,
    bereich_df,
    soft_df,
    sprachen_df,
    fuehr_df,
    arbeitszeit_df,
], ignore_index=True)

all_keywords.to_csv("logistiker_all_keywords.csv", index=False, encoding="utf-8-sig")

print(f"\nGesamt Keywords: {len(all_keywords)} Zeilen")
print(f"\nKategorien:")
print(all_keywords["Kategorie"].value_counts())

Stellen vorher: 2733

Verteilung nach Jahren (vorher):
Veroeffentlicht
2025     188
2026    2545
Name: count, dtype: int64

Stellen nachher: 2733
Zeitraum: 2025-06-02 — 2026-06-22
logistiker_jobs_clean.csv gespeichert!

Gesamt Keywords: 1720 Zeilen

Kategorien:
Kategorie
Hard_Skills      540
Soft_Skills      291
Tools            282
Arbeitszeit      224
Sprachen         218
Bereich          128
Fuehrerschein     37
Name: count, dtype: int64
